In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.express as px

In [2]:
DB_PATH = Path("./clinical.db").resolve()
print("DB_PATH:", DB_PATH)
print("Exists:", DB_PATH.exists())
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

DB_PATH: /Users/cleberfcarvalho/Documents/myGitHub/clinical-sql-analytics-diabetes-readmissions/clinical.db
Exists: True


In [ ]:
query_age = """
SELECT
p.age,
ROUND(
100.0 * SUM(CASE WHEN e.readmitted <> 'NO' THEN 1 ELSE 0 END)
/ COUNT(*),2
) AS readmission_rate
FROM encounters e
JOIN patients p
ON e.patient_nbr = p.patient_nbr
GROUP BY p.age
ORDER BY p.age;
"""

df_age = pd.read_sql_query(query_age, conn)

fig = px.line(
    df_age,
    x="age",
    y="readmission_rate",
    markers=True,
    title="Readmission Rate by Age Group"
)
fig.write_image("./reports/readmission_age.png", scale=2)
fig.show()



In [4]:
query_risk = """
SELECT
CASE
WHEN number_diagnoses >= 9 THEN 'High'
WHEN number_diagnoses BETWEEN 6 AND 8 THEN 'Medium'
ELSE 'Low'
END AS risk_group,
ROUND(
100.0 * SUM(CASE WHEN readmitted <> 'NO' THEN 1 ELSE 0 END)
/ COUNT(*),2
) AS readmission_rate
FROM encounters
GROUP BY risk_group;
"""

df_risk = pd.read_sql_query(query_risk, conn)
fig = px.bar(
    df_risk,
    x="risk_group",
    y="readmission_rate",
    title="Readmission Rate by Clinical Risk Group",
    text="readmission_rate"
)
fig.write_html("./reports/readmission_clinical_risk_group.png")


fig.show()

In [5]:
query_a1c = """
SELECT
CASE
    WHEN a1c_result = 'Norm' THEN 'Normal'
    WHEN a1c_result = '>7' THEN 'Elevated'
    WHEN a1c_result = '>8' THEN 'Poor Control'
    ELSE 'Not Measured'
END AS glycemic_status,
ROUND(
100.0 * SUM(CASE WHEN readmitted <> 'NO' THEN 1 ELSE 0 END)
/ COUNT(*),2
) AS readmission_rate
FROM encounters
GROUP BY glycemic_status;
"""

df_a1c = pd.read_sql_query(query_a1c, conn)

fig = px.bar(
    df_a1c,
    x="glycemic_status",
    y="readmission_rate",
    title="Readmission Rate by Glycemic Control",
    text="readmission_rate"
)
fig.write_html("./reports/readmission_rate_glycemic_control.png")


fig.show()